# Goal

Шаг  новой цепи на `18n_ppo_tr_frostbite_08` (окно из 8 обзов): обучение от чемпионов `18n_study_38.3`. 

Просто даём играть на пятом уровне - продавим или нет? **Сознательно не берём во внимание вопрос забывания прошлого опыта - сможет ли в прицнипе одолеть пятый уровень?**

`decisive_metric=rollout/episode_r`

# TARGET_NOTEBOOK_FNAME

In [31]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_08.ipynb'

# GRID_SEARCH_SPACE

In [32]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [33]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 200)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    HP.general.is_torch_amp_on_rollout = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.vision_head.parent = dict(model='18d_world_model_09:183', weights='18d_world_model_09:183')
    HP.vision_head.is_trainable = False
    
    HP.encoder.parent = dict(model='18d_world_model_09:183', weights='18d_world_model_09:183')
    HP.encoder.is_trainable = True

    parent = '18n_ppo_tr_frostbite_08:98'
    HP.encoder.parent['weights'] = parent
    HP.agent.parent = parent
    
    HP.agent.sequence_length = 33
    HP.agent.sequence_ages = [0, 1, 2, 3, 8, 16, 24, 32]
    HP.agent.action_plan_length = 10
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = [
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none', 
    ]
    HP.test.env_ram_patches = [
        ['three_lives', 'no_score', 'no_igloo', 'temperature_45'], # standard game start
    ]
    HP.test.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 1_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none1', 
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none2', 
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none3', 
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none4', 
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none5', 
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none6', 
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none7', 
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none8', 
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none9', 
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none10',
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['three_lives', 'no_score', 'no_igloo', 'temperature_45'], # standard game start
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 256 
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.value_loss = dict(coef=optuna_trial.suggest_float('value_loss_coef', 0.01, 0.5))
    entropy_loss_coef = optuna_trial.suggest_float('entropy_loss_coef', 0.01, 0.5)
    HP.ppo.entropy_loss = dict(coef=f'const({entropy_loss_coef})')
    HP.ppo.consistency_loss = dict(type='MSE', coef=optuna_trial.suggest_float('consistency_loss_coef', 0.01, 0.5))
    HP.ppo.prediction_loss = dict(coef=optuna_trial.suggest_float('prediction_loss_coef', 0.01, 0.5))
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization

    HP.ppo.follow_up_freq = 2
    return HP

# Results

Не, полная хрень. Ни одного пробития нет. В дискуссии с Риком стало понятно, что все эти коэффициенты мало на что влияют, если агент не получает advantage - так всё на уровне шума.

<img src="./img/episode_r_38.4b.png">
<img src="./img/full_picture.png">

**Выводы**
1) неужели дело в 183-ем `VisionHead`?

# System

In [34]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [35]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_08.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_08',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_38.4b/18n_study_38.4b.ipynb',
 'optuna_study_name': '18n_study_38.4b',
 'optuna_study_serial': '38.4b',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_38.4b/18n_study_38.4b.optuna'}

In [36]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [37]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [38]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC='rollout/episode_r',
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [39]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [40]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 50
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-12 14:56:52,084] A new study created in Journal with name: 18n_study_38.4b


2026.10.12-14:56:52.505512     0.401 >> Model instance registered, version=133
2026.10.12-14:56:52.528144     0.010 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_08-launch133.ipynb"
2026.10.12-14:56:52.529388     0.011 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_08:133"; running launches=1
2026.10.12-14:57:24.100198     0.392 >> Model instance registered, version=134
2026.10.12-14:57:24.114830     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_08-launch134.ipynb"
2026.10.12-14:57:24.115171     0.006 >> 5.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_08:134"; running launches=2
2026.10.12-14:57:55.191350     0.320 >> Model instance registered, version=135
2026.10.12-14:57:55.202413     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_08-launch135.ipynb"
2026.10.12-14:57:55.202823     0.005 >> 4.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_08

In [41]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-12 16:48:24,679] Using an existing study with name '18n_study_38.4b' instead of creating a new one.


2026.10.12-16:48:24.680564     5.040 >> Study statistics: 
2026.10.12-16:48:24.687569     0.007 >> 	Number of finished trials: 50
2026.10.12-16:48:24.690202     0.003 >> 	Number of pruned trials: 0
2026.10.12-16:48:24.691815     0.002 >> 	Number of complete trials: 50
2026.10.12-16:48:24.692645     0.001 >> Best trial:
2026.10.12-16:48:24.693848     0.001 >> 	Value: 1322.1875
2026.10.12-16:48:24.694216     0.000 >> 	Model version: 141
2026.10.12-16:48:24.694855     0.001 >> 	Params: 
2026.10.12-16:48:24.695274     0.000 >> 		value_loss_coef: 0.08582488420844027
2026.10.12-16:48:24.695752     0.000 >> 		entropy_loss_coef: 0.11640487499559131
2026.10.12-16:48:24.696100     0.000 >> 		consistency_loss_coef: 0.18211473737733935
2026.10.12-16:48:24.696389     0.000 >> 		prediction_loss_coef: 0.27078497570649507


In [42]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.12-16:48:24.709857     0.013 >> Top trials:
2026.10.12-16:48:24.710634     0.001 >> 1) 18n_ppo_tr_frostbite_08:141, value=1322.1875
2026.10.12-16:48:24.710966     0.000 >> 2) 18n_ppo_tr_frostbite_08:145, value=1304.6875
2026.10.12-16:48:24.711317     0.000 >> 3) 18n_ppo_tr_frostbite_08:177, value=1300.6451612903227


In [43]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(1240.4452682710048), np.float64(33.07376259256854))

In [44]:
','.join(map(str, values.tolist()))

'1228.3870967741937,1229.0625,1248.4848484848485,1224.2424242424242,1255.0,1228.125,1276.1764705882354,1177.9411764705883,1322.1875,1248.1818181818182,1242.1875,1295.7575757575758,1304.6875,1218.4375,1212.121212121212,1224.5454545454545,1263.4375,1231.9444444444443,1219.142857142857,1298.1818181818182,1284.375,1263.939393939394,1234.4117647058824,1205.625,1282.7586206896551,1238.75,1196.5625,1237.5757575757575,1201.5625,1253.235294117647,1231.6666666666667,1240.909090909091,1239.7058823529412,1233.030303030303,1214.2857142857142,1207.8125,1195.8823529411766,1162.162162162162,1281.25,1242.121212121212,1239.375,1254.6875,1254.8484848484848,1245.0,1300.6451612903227,1254.6875,1201.6666666666667,1204.6875,1215.4545454545455,1255.357142857143'